# Coding Our Problem For Minimizing Risk

This implementation of Markowitz Mean-Variance Optimization uses the package `cvxpy`, which is meant for convex optimization problems (like what we are solving). You can find the documentation for this package [here](https://www.cvxpy.org/).

Also, read [this](https://ranaroussi.github.io/yfinance/) if you want a more comprehensive view of the data being imported.

<br>

**NOTE: All of our units for these problems is DAILY because we are important daily data from the package `yfinance`.**

<br>

**Steps for Implementation:**
1. Import our historical adjusted closing prices and convert that to historical rate of return data.
2. Use our data to calculate an estimated covariance matrix $\hat{\mathbf{S}}$.
3. Estimate our vector of expected rates of returns $\mathbf{m}$ as $\hat{\mathbf{m}}=[\bar{r}_1,\bar{r}_2,\cdots,\bar{r}_n]^T$ (remember $\bar{r}_i$ are the column means from our data matrix $\mathbf{X}$, a.k.a. the mean rate of return for asset $i$, $1\leq i \leq n$).
4. Plug our estimates $\hat{\mathbf{S}}$ and $\hat{\mathbf{m}}$ into the optimization problem we defined, and solve for our optimal set of weights $\mathbf{w^*}$ using `cvxpy`'s powerful solver.

Recall the mathematical statement of our optimization problem:

\begin{align*}
    \text{minimize } & \frac{1}{2} \mathbf{w}^T\mathbf{S}\mathbf{w} \\
    \text{subject to the constraints } & \mathbf{m}^T\mathbf{w}\geq\mu_b \text{ and } \mathbf{1}^T\mathbf{w}=1
\end{align*}


In [12]:
# this is a package for solving convex optimization problems
import cvxpy as cp
import numpy as np
import yfinance as yf
import pandas as pd

def solve_markowitz(cov_matrix, expected_ror, baseline_ror):
    n = len(expected_ror)

    # defining the variable to be optimized
    w = cp.Variable(n)

    # minimizing portfolio variance (0.5 * w^T * S * w)
    # cp.quad_form defines "quadratic terms" like w^T * S * w
    objective = cp.Minimize(0.5 * cp.quad_form(w, cov_matrix))

    # defining the constraints
    constraints = [
        cp.sum(w) == 1, # budget constraint: 1^T * w = 1
        expected_ror @ w >= baseline_ror # baseline return constraint: m^T * w >= mu_b
    ]

    # solve the problem using cvxpy
    prob = cp.Problem(objective, constraints)
    prob.solve(solver = cp.CLARABEL)

    # debugging
    if prob.status not in ("optimal","optimal_inaccurate"):
      raise ValueError(f"Solver status: {prob.status} (no solution)")

    return w.value

# DRIVER CODE
# import data
# this is a list of the companies we are extracting stock closes for
tickers = ["AAPL", "MSFT", "GOOG","NVDA","META"]
data = yf.download(tickers,start = "2015-01-01",
                   end = "2025-01-01",
                   auto_adjust = False,
                   progress= False)

# extracting just the adjusted close values
data = data["Adj Close"]

# using pandas to compute the percent change between current row and prior row (for every row)
# these are our rates of return (remember, RoR = (x1-x0)/x0) where x1 is selling price and x0 is buying price
# dropping the first row of NANs (there is no percent change to calculate for the first row)
data = data.pct_change().dropna()

# finding our sample covariance matrix
# rowvar = False indicates that rows are observations and columns are variables
sample_cov_matrix = np.cov(m=data,rowvar=False)

# column vector of expected rates of return
sample_means = data.mean().values

# this is our mu_b (baseline expected rate of return)
baseline_ror = 0.002

# calling our function to find our optimal weights
solution = solve_markowitz(cov_matrix=sample_cov_matrix,
                    expected_ror=sample_means,
                    baseline_ror=baseline_ror)

# printing our output
print("--- Optimal Portfolio Weights Using Markowitz MVO ---")
for name, wt in zip(data.columns, solution):
       print(f"{name} Weight: {wt*100:.4f}%")

# making sure our chosen weights satisfy the constraints
print(f"Total Weights (should equal 1): {np.sum(solution):.2f}")
portfolio_variance = np.dot(solution, sample_cov_matrix @ solution).item()
print(f"Variance of ROR on Portfolio Using Optimal Weights: {portfolio_variance:.3e}")
print(f"Expected ROR on Portfolio Using Optimal Weights (should be at least {baseline_ror}): {np.dot(solution, sample_means).item():.3e}")


--- Optimal Portfolio Weights Using Markowitz MVO ---
AAPL Weight: 25.0375%
GOOG Weight: 3.3150%
META Weight: -3.6568%
MSFT Weight: 18.5067%
NVDA Weight: 56.7976%
Total Weights (should equal 1): 1.00
Variance of ROR on Portfolio Using Optimal Weights: 4.979e-04
Expected ROR on Portfolio Using Optimal Weights (should be at least 0.002): 2.000e-03


Try changing the baseline expected rate of return `baseline_ror` in the code above. Make it super low then make it super high.
- What happens? Can you explain **why** that happens in financial terms?
- Is your overall portfolio getting more or less volatile the higher you make `baseline_ror`?
- When is it encouraging or discouraging shorting your assets (and if so, which assets)?
- Does the expected rate of return on your portfolio stop changing if you make `baseline_ror` too low? What does this say about the value we are minimizing (the variance), does it have some sort of "global" minimum that you can't go below?

# Coding our Problem in `cplex`

The package `cplex` is commonly used in industry to solve these types of problems, so here's some code that solves this same problem in `cplex` in case you are interested!

In [9]:
# let's first install cplex
!pip install cplex

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 MB 18.1 MB/s eta 0:00:00


In [14]:
import cplex
import numpy as np
import yfinance as yf

def markowitz_cplex(p, means, assets, cov_matrix, baseline_ror):
    num_asset = len(assets)

    # minimize portfolio variance / risk
    p.objective.set_sense(p.objective.sense.minimize)

    # add the number of variables as you have assets in your portfolio
    names = ['w'+str(i) for i in range(num_asset)]

    # adding the constraints
    p.variables.add(obj = [0]*num_asset,
                    ub = [cplex.infinity]*num_asset, # setting "upper bound" to infinity to allow for a weight as high as we want
                    lb = [-cplex.infinity]*num_asset, # setting "lower bound" to negative infinity to allow infinite short selling
                    # NOTE: If we don't want infinite short selling, we can set these upper and lower bounds to something specific like 0 or 1
                    names = names)

    # 'E' stands for "equal" and 'G' stands for "greater (than or equal to)"
    my_senses = 'EG'
    my_rhs = [1,baseline_ror]
    rows = [[names, [1]*num_asset],  # The sum of all weights is 1
           [names, list(means)]]     # The expected return is larger than the baseline


    p.linear_constraints.add(lin_expr = rows, senses = my_senses, rhs = my_rhs)

    # add quadratic part in the objective
    qmat = []
    for j in range(num_asset):
        qmat.append([[i for i in range(num_asset)],
          # cplex automatically appends a 1/2* to quadratic terms, by quadratic programming convention!
         [cov_matrix[j, m] for m in range(num_asset)]])

    p.objective.set_quadratic(qmat)

# DRIVER CODE
tickers = ["AAPL", "MSFT", "GOOG","NVDA","META"]
data = yf.download(tickers,start = "2015-01-01",
                   end = "2025-01-01",
                   auto_adjust = False,
                   progress= False)

data = data["Adj Close"]
data = data.pct_change().dropna()

# reordering the tickers to align with the data
tickers = list(data.columns)

sample_cov_matrix = np.cov(m=data,rowvar=False)
sample_means = data.mean().values
baseline_ror = 0.002

p = cplex.Cplex()
# ensuring our output is within a certain accuracy
p.parameters.barrier.convergetol.set(1e-12)
markowitz_cplex(p=p,
               means=sample_means,
               assets=tickers,
               cov_matrix=sample_cov_matrix,
                baseline_ror=baseline_ror)

# suppressing the messages that show when you solve the system
p.set_log_stream(None)
p.set_results_stream(None)
p.solve()

# printing our output
print("\n--- Optimal Portfolio Weights Using Markowitz in cplex ---")
optimal_weights = p.solution.get_values()
for i, ticker in enumerate(tickers):
  print(f"{ticker}: {optimal_weights[i]*100:.4f}%")
print(f"Total Weights (should equal 1): {sum(p.solution.get_values()):.2f}")
print(f"Variance of ROR on Portfolio Using Optimal Weights: {2*p.solution.get_objective_value():.3e}")
print(f"Expected ROR on Portfolio Using Optimal Weights (should be at least {baseline_ror}): {np.dot(p.solution.get_values(),sample_means):.3e}")


--- Optimal Portfolio Weights Using Markowitz in cplex ---
AAPL: 25.0376%
GOOG: 3.3151%
META: -3.6568%
MSFT: 18.5068%
NVDA: 56.7974%
Total Weights (should equal 1): 1.00
Variance of ROR on Portfolio Using Optimal Weights: 4.979e-04
Expected ROR on Portfolio Using Optimal Weights (should be at least 0.002): 2.000e-03


This code is a good sanity check for our other solution!

- Try out different values of `baseline_ror` and see if the outputs for the code using `cvxpy` from before and this code using `cplex` match.

# Exercise: Try To Code Our Problem For Maximizing Reward

Use the code below as a template to code the second problem we defined, where our overall goal was to maximize return while constraining our risk:


\begin{align*}
    \text{maximize } & \mathbf{m}^T\mathbf{w} \\
    \text{subject to the constraints } & \frac{1}{2} \mathbf{w}^T\mathbf{S}\mathbf{w}\leq\sigma^2_{max} \text{ and } \mathbf{1}^T\mathbf{w}=1
\end{align*}


Once you are done with modifying the code below to solve this new problem, you can [go here to see the solution](https://colab.research.google.com/drive/1H4QK3uyW-qIpkh_mKEBCEobxass1jlin?usp=sharing)! Additionally, once you are done coding, try the exploration tasks below!

<br>

**WARNING: The code block below will have errors if you run it now. You must complete the code by filling in the code for the objective and constraint functions!**

In [15]:
import cvxpy as cp
import numpy as np
import yfinance as yf
import pandas as pd

def solve_markowitz(cov_matrix, expected_ror, max_variance):
    n = len(expected_ror)
    w = cp.Variable(n)

    # objective function
    objective = cp.Maximize("expression here!")

    # defining the constraints
    constraints = [
        "constraints here!"
      ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver = cp.CLARABEL)

    # debugging
    if prob.status not in ("optimal","optimal_inaccurate"):
      raise ValueError(f"Solver status: {prob.status} (no solution)")

    return w.value

# DRIVER CODE
# import data
tickers = ["AAPL", "MSFT", "GOOG","NVDA","META"]
data = yf.download(tickers,start = "2015-01-01",
                   end = "2025-01-01",
                   auto_adjust = False,
                   progress = False)
data = data["Adj Close"]
data = data.pct_change().dropna()

# sample covariance matrix
sample_cov_matrix = np.cov(m=data,rowvar=False)

# expected rates of return
sample_means = data.mean().values

# this is our sigma^2_max (maximum limit on 1/2 of the variance of our portfolio ROR)
# remember, our actual variance will be capped at 2 times the amount set here due to the 1/2 factor out front
max_variance = 0.0002

# finding our optimal weights
solution = solve_markowitz(cov_matrix=sample_cov_matrix,
                    expected_ror=sample_means,
                    max_variance=max_variance)

# printing our output
print("--- Optimal Portfolio Weights Using Markowitz MVO ---")
for name, wt in zip(data.columns, solution):
       print(f"{name} Weight: {wt*100:.4f}%")

# making sure our chosen weights satisfy the constraints
print(f"Total Weights (should equal 1): {np.sum(solution):.2f}")
print(f"Expected ROR on Portfolio Using Optimal Weights: {np.dot(solution, sample_means).item():.3e}")
portfolio_variance = np.dot(solution, sample_cov_matrix @ solution).item()
print(f"Portfolio Variance (should be less than or equal to 2 * {max_variance} = {2*max_variance}): {portfolio_variance:.3e}")

ValueError: could not convert string to float: np.str_('expression here!')

## Guided Exploration Task 1

Try changing `data` to be the following toy example:

$$
\begin{bmatrix}
2 & 2 \\
2&2 \\
5&5
\end{bmatrix}
$$

In code, you would write `data = pd.DataFrame([2,2],[2,2],[5,5]), columns = ["A","B"])` and put it right after the line `data = data.pct_change().dropna()`.

Change `max_variance = 1.4` as well.

<br>

---

**Does the solver raise an error?** The answer is yes! **But why?**

---

<br>

Notice that here we have 2 assets which are **perfectly correlated**! Their `sample_cov_matrix` will be equal to:

$$
\begin{bmatrix}
3 & 3 \\
3 & 3
\end{bmatrix}
$$

This is important because let's take a closer look at our portfolio variance $\mathbf{w}^T\mathbf{S}\mathbf{w}$


\begin{align*}
    \mathbf{w}^T\mathbf{S}\mathbf{w}
    = \sum_{j=1}^n\sum_{i=1}^nw_jw_iCov(r_j,r_i)
    = 3\sum_{j=1}^n\sum_{i=1}^nw_jw_i
\end{align*}

Since $Cov(r_j,r_i)=3$ for all pairs $(i,j)$.

Remember that our constraint is that $\frac{1}{2}$ the portfolio variance, that is $\frac{1}{2}\mathbf{w}^T\mathbf{S}\mathbf{w}$, cannot exceed $\sigma^2_{max}=1.4$:

\begin{align*}
    \frac{3}{2}\sum_{j=1}^n\sum_{i=1}^nw_jw_i \leq 1.4
\end{align*}

Choose different values of weights that add to 1 and you'll start to notice a problem here. No matter *what* weights we choose (whether they be drastically negative or positive), it will **always** fail to satisfy this variance constraint! Let's demonstrate with the weight vector $w=[-k,k+1]^T,k \in \mathbb{R}$ (remember, our weights have to add to $1$ by our budget constraint):

\begin{align*}
    \frac{3}{2}\sum_{j=1}^n\sum_{i=1}^nw_jw_i
    &= \frac{3}{2}\Big((-k)^2+(k+1)^2+2(-k)(k+1)\Big) \\
    &= \frac{3}{2}(1) \\
    &= 1.5 \not\leq 1.4
\end{align*}

This causes our solver to raise an error because no matter what weights we choose, our problem is **infeasible** and has no solution that satisfies our given constraints! This causes our solver to return a solution of `None`, hence our error.

Try changing `max_variance = 1.5`, and you'll notice that the compiler no longer raises an error. Our problem becomes **feasible** again!

This demonstrates to us an important general fact: sometimes the problem we pose can be **infeasible**, or not concretely solvable for some data sets because our constraints are mathematically impossible to satisfy all at once.

**NOTE: For our data imported from `yfinance`, our infeasiblility borderline is at about `max_variance = 0.0001`!**








## DIY Exploration Task 2

Building off our last example, keep `data` to be the following:

$$
\begin{bmatrix}
2 & 2 \\
2&2 \\
5&5
\end{bmatrix}
$$

And change `max_variance = 1.5` (to make our problem feasible). This time, manually overwrite `sample_means = data.mean().values` to now be change `sample_means = np.array([1,2])`.

<br>

---

**Does the solver raise an error this time? Yes again! This time, can you explain why? The error message says `unbounded`, what could that mean?**

---

<br>

**Here's a hint**: the answer has to do with shorting an asset and the fact our variance constraint is always satisfied.

<br>

---

**Once you've figured out why this error is occurring, can you come up with a solution?**

---

<br>

**Another Hint**: If shorting an asset is the problem, can we constrain our problem to *not* allow it? Remember, **we** are the ones who make the mathematical optimization problem, so we can constrain whatever we want! That's part of the beauty (and power) of understanding the math behind the algorithm.